# Phase 2: Huấn luyện và mô phỏng World Cup 2026

Notebook này đánh giá model theo thời gian, refit bằng toàn bộ trận đã đá, rồi mô phỏng vòng bảng bằng tỷ số để có thể xử lý tie-break.

In [165]:
from collections import defaultdict
from itertools import groupby

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.metrics import accuracy_score, log_loss, mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df_recent = pd.read_csv("data/processed/matches_features.csv", parse_dates=["date"])
upcoming_matches = pd.read_csv("data/processed/upcoming_matches.csv", parse_dates=["date"])

assert df_recent["outcome"].notna().all()
assert upcoming_matches["outcome"].isna().all()
assert df_recent.duplicated().sum() == 0

print("Trận đã đá:", df_recent.shape)
print("Trận chưa đá:", upcoming_matches.shape)

Trận đã đá: (10784, 19)
Trận chưa đá: (32, 19)


## Đánh giá model theo thời gian

Các trận trước năm 2024 dùng để train; các trận từ năm 2024 trở đi dùng để test. Elo của mỗi dòng đã được ghi **trước trận**, nên phép đánh giá tương ứng với dự báo tuần tự theo thời gian.

In [166]:
CUTOFF_DATE = pd.Timestamp("2024-01-01")
train = df_recent[df_recent["date"].lt(CUTOFF_DATE)].copy()
test = df_recent[df_recent["date"].ge(CUTOFF_DATE)].copy()

print("Train:", train.shape)
print("Test:", test.shape)

RECENCY_HALF_LIFE_YEARS = 4
FORM_FEATURES = [
    "home_points_form", "away_points_form",
    "home_gf_form", "away_gf_form",
    "home_ga_form", "away_ga_form",
]

def recency_weights(dates):
    age_years = (dates.max() - dates).dt.days / 365.25
    return 0.5 ** (age_years / RECENCY_HALF_LIFE_YEARS)

train_weights = recency_weights(train["date"])
full_weights = recency_weights(df_recent["date"])

Train: (8872, 19)
Test: (1912, 19)


In [167]:
def make_classifier():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=2000),
    )


def classifier_classes(model):
    return model.named_steps["logisticregression"].classes_


candidate_features = {
    "elo_only": ["elo_diff"],
    "elo_and_neutral": ["elo_diff", "neutral"],
    "elo_neutral_and_form": ["elo_diff", "neutral"] + FORM_FEATURES,
}

evaluation_rows = []
validation_models = {}
subsets = {
    "all_test": pd.Series(True, index=test.index),
    "neutral_test": test["neutral"].eq(True),
    "world_cup_test": test["tournament"].eq("FIFA World Cup"),
}

for model_name, features in candidate_features.items():
    fitted = make_classifier()
    fitted.fit(
        train[features],
        train["outcome"],
        logisticregression__sample_weight=train_weights,
    )
    validation_models[model_name] = fitted
    classes = classifier_classes(fitted)

    for subset_name, mask in subsets.items():
        subset = test.loc[mask]
        probabilities = fitted.predict_proba(subset[features])
        predictions = fitted.predict(subset[features])
        evaluation_rows.append({
            "model": model_name,
            "subset": subset_name,
            "n_matches": len(subset),
            "accuracy": accuracy_score(subset["outcome"], predictions),
            "log_loss": log_loss(subset["outcome"], probabilities, labels=classes),
        })

evaluation = pd.DataFrame(evaluation_rows)
evaluation

,model,subset,n_matches,accuracy,log_loss
0,elo_only,all_test,1912,0.595711,0.879457
1,elo_only,neutral_test,700,0.537143,0.964931
2,elo_only,world_cup_test,40,0.475000,1.003245
3,elo_and_neutral,all_test,1912,0.592050,0.879692
4,elo_and_neutral,neutral_test,700,0.540000,0.964542
5,elo_and_neutral,world_cup_test,40,0.500000,1.001096
6,elo_neutral_and_form,all_test,1912,0.596234,0.871229
7,elo_neutral_and_form,neutral_test,700,0.552857,0.944149
8,elo_neutral_and_form,world_cup_test,40,0.500000,0.976585


In [168]:
class_order = sorted(train["outcome"].unique())
train_prior = train["outcome"].value_counts(normalize=True).reindex(class_order)
baseline_probabilities = np.tile(train_prior.to_numpy(), (len(test), 1))
baseline_prediction = train_prior.idxmax()

print("Baseline accuracy:", round((test["outcome"] == baseline_prediction).mean(), 3))
print("Baseline log loss:", round(log_loss(test["outcome"], baseline_probabilities, labels=class_order), 3))

Baseline accuracy: 0.468
Baseline log loss: 1.057


## Model phân loại cuối cùng

Model cuối giữ `neutral` và form 5 trận gần nhất vì chúng cải thiện log loss trên tập test theo thời gian, đặc biệt ở sân trung lập. Dữ liệu cũ được giảm trọng số với half-life 4 năm. Sau khi đánh giá, model được refit bằng **toàn bộ trận đã đá**.

In [169]:
FINAL_FEATURES = ["elo_diff", "neutral"] + FORM_FEATURES
final_model = make_classifier()
final_model.fit(
    df_recent[FINAL_FEATURES],
    df_recent["outcome"],
    logisticregression__sample_weight=full_weights,
)


def predict_outcome_probabilities(model, row, features=FINAL_FEATURES):
    feature_row = pd.DataFrame([{feature: row[feature] for feature in features}])
    probabilities = model.predict_proba(feature_row)[0]
    return dict(zip(classifier_classes(model), probabilities))

## Model tỷ số cho Monte Carlo

Logistic Regression chỉ tạo kết quả thắng/hòa/thua nên không đủ để xếp hạng khi bằng điểm. Hai Poisson Regression dưới đây dự đoán số bàn của đội nhà và đội khách; nhờ đó mô phỏng có thể tính hiệu số và bàn thắng.

In [170]:
GOAL_FEATURES = FINAL_FEATURES


def make_goal_model():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        PoissonRegressor(alpha=0.1, max_iter=2000),
    )


home_goal_validation = make_goal_model()
away_goal_validation = make_goal_model()
home_goal_validation.fit(
    train[GOAL_FEATURES], train["home_score"],
    poissonregressor__sample_weight=train_weights,
)
away_goal_validation.fit(
    train[GOAL_FEATURES], train["away_score"],
    poissonregressor__sample_weight=train_weights,
)

home_mae = mean_absolute_error(
    test["home_score"], home_goal_validation.predict(test[GOAL_FEATURES])
)
away_mae = mean_absolute_error(
    test["away_score"], away_goal_validation.predict(test[GOAL_FEATURES])
)
print("MAE bàn đội nhà:", round(home_mae, 3))
print("MAE bàn đội khách:", round(away_mae, 3))

home_goal_model = make_goal_model()
away_goal_model = make_goal_model()
home_goal_model.fit(
    df_recent[GOAL_FEATURES], df_recent["home_score"],
    poissonregressor__sample_weight=full_weights,
)
away_goal_model.fit(
    df_recent[GOAL_FEATURES], df_recent["away_score"],
    poissonregressor__sample_weight=full_weights,
)

MAE bàn đội nhà: 1.059
MAE bàn đội khách: 0.882


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('simpleimputer', ...), ('standardscaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['elo_diff','neutral','home_points_form',...,'away_gf_form','home_ga_form', 'away_ga_form']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" 

In [171]:
wc_played_all = df_recent[
    df_recent["tournament"].eq("FIFA World Cup")
    & df_recent["date"].ge("2026-01-01")
].copy()
wc_upcoming_all = upcoming_matches[
    upcoming_matches["tournament"].eq("FIFA World Cup")
].copy()

# 72 trận đầu tiên của World Cup 2026 là vòng bảng.
wc_group_matches = (
    pd.concat([wc_played_all, wc_upcoming_all], ignore_index=True, sort=False)
    .sort_values(["date", "home_team", "away_team"])
    .head(72)
    .reset_index(drop=True)
)
wc2026_played = wc_group_matches.dropna(subset=["home_score", "away_score"]).copy()
wc2026_upcoming = wc_group_matches[
    wc_group_matches[["home_score", "away_score"]].isna().any(axis=1)
].copy()

assert len(wc_group_matches) == 72
assert wc2026_upcoming["tournament"].eq("FIFA World Cup").all()
print("Đã đá:", len(wc2026_played), "| Chưa đá:", len(wc2026_upcoming))

Đã đá: 40 | Chưa đá: 32


In [172]:
opponents = defaultdict(set)
for row in wc_group_matches.itertuples(index=False):
    opponents[row.home_team].add(row.away_team)
    opponents[row.away_team].add(row.home_team)

unvisited = set(opponents)
groups = []
while unvisited:
    start = unvisited.pop()
    component = {start}
    stack = [start]
    while stack:
        team = stack.pop()
        new_teams = opponents[team] - component
        component.update(new_teams)
        unvisited.difference_update(new_teams)
        stack.extend(new_teams)
    groups.append(tuple(sorted(component)))

groups = sorted(groups, key=lambda group: group[0])
assert len(groups) == 12 and all(len(group) == 4 for group in groups)
for number, group in enumerate(groups, start=1):
    print(number, group)

1 ('Algeria', 'Argentina', 'Austria', 'Jordan')
2 ('Australia', 'Paraguay', 'Turkey', 'United States')
3 ('Belgium', 'Egypt', 'Iran', 'New Zealand')
4 ('Bosnia and Herzegovina', 'Canada', 'Qatar', 'Switzerland')
5 ('Brazil', 'Haiti', 'Morocco', 'Scotland')
6 ('Cape Verde', 'Saudi Arabia', 'Spain', 'Uruguay')
7 ('Colombia', 'DR Congo', 'Portugal', 'Uzbekistan')
8 ('Croatia', 'England', 'Ghana', 'Panama')
9 ('Curaçao', 'Ecuador', 'Germany', 'Ivory Coast')
10 ('Czech Republic', 'Mexico', 'South Africa', 'South Korea')
11 ('France', 'Iraq', 'Norway', 'Senegal')
12 ('Japan', 'Netherlands', 'Sweden', 'Tunisia')


In [173]:
def predict_goal_rates(row):
    features = pd.DataFrame([{feature: row[feature] for feature in GOAL_FEATURES}])
    home_rate = float(home_goal_model.predict(features)[0])
    away_rate = float(away_goal_model.predict(features)[0])
    return np.clip(home_rate, 0.05, 7.0), np.clip(away_rate, 0.05, 7.0)


goal_rates = wc2026_upcoming.apply(predict_goal_rates, axis=1, result_type="expand")
goal_rates.columns = ["home_goal_rate", "away_goal_rate"]
wc2026_upcoming = pd.concat(
    [wc2026_upcoming.reset_index(drop=True), goal_rates.reset_index(drop=True)],
    axis=1,
)

probability_rows = []
for _, row in wc2026_upcoming.iterrows():
    probability_rows.append({
        "date": row["date"],
        "home_team": row["home_team"],
        "away_team": row["away_team"],
        **predict_outcome_probabilities(final_model, row),
    })
match_probabilities = pd.DataFrame(probability_rows)
match_probabilities.head()

,date,home_team,away_team,away_win,draw,home_win
0,2026-06-22,Argentina,Austria,0.115453,0.234359,0.650188
1,2026-06-22,France,Iraq,0.097639,0.207114,0.695246
2,2026-06-22,Jordan,Algeria,0.309979,0.305060,0.384961
3,2026-06-22,Norway,Senegal,0.356256,0.276397,0.367347
4,2026-06-23,Colombia,DR Congo,0.257590,0.307909,0.434501


## Xếp hạng và tie-break

Thứ tự được xét theo điểm, hiệu số và bàn thắng; nếu vẫn bằng nhau thì xét thành tích đối đầu giữa các đội đang hòa. Dữ liệu không có điểm fair-play, nên trường hợp vẫn hòa tuyệt đối được xử lý bằng bốc thăm có seed thay vì phụ thuộc vào thứ tự ngẫu nhiên của `set`.

In [174]:
def calculate_table(teams, results):
    table = {team: {"team": team, "points": 0, "gf": 0, "ga": 0} for team in teams}
    for home, away, home_goals, away_goals in results:
        if home not in table or away not in table:
            continue
        table[home]["gf"] += home_goals
        table[home]["ga"] += away_goals
        table[away]["gf"] += away_goals
        table[away]["ga"] += home_goals
        if home_goals > away_goals:
            table[home]["points"] += 3
        elif home_goals < away_goals:
            table[away]["points"] += 3
        else:
            table[home]["points"] += 1
            table[away]["points"] += 1

    for row in table.values():
        row["gd"] = row["gf"] - row["ga"]
    return table


def rank_group(teams, results, rng):
    overall = calculate_table(teams, results)
    rows = sorted(
        overall.values(),
        key=lambda row: (row["points"], row["gd"], row["gf"]),
        reverse=True,
    )

    ranked = []
    key_function = lambda row: (row["points"], row["gd"], row["gf"])
    for _, tied_rows in groupby(rows, key=key_function):
        tied_rows = list(tied_rows)
        if len(tied_rows) == 1:
            ranked.extend(tied_rows)
            continue

        tied_teams = {row["team"] for row in tied_rows}
        head_to_head = calculate_table(tied_teams, results)
        lottery = {team: rng.random() for team in tied_teams}
        tied_rows.sort(
            key=lambda row: (
                head_to_head[row["team"]]["points"],
                head_to_head[row["team"]]["gd"],
                head_to_head[row["team"]]["gf"],
                lottery[row["team"]],
            ),
            reverse=True,
        )
        ranked.extend(tied_rows)
    return ranked

In [175]:
group_data = []
for group in groups:
    played = wc2026_played[
        wc2026_played["home_team"].isin(group)
        & wc2026_played["away_team"].isin(group)
    ]
    future = wc2026_upcoming[
        wc2026_upcoming["home_team"].isin(group)
        & wc2026_upcoming["away_team"].isin(group)
    ]
    played_results = [
        (row.home_team, row.away_team, int(row.home_score), int(row.away_score))
        for row in played.itertuples(index=False)
    ]
    future_matches = [
        (row.home_team, row.away_team, row.home_goal_rate, row.away_goal_rate)
        for row in future.itertuples(index=False)
    ]
    group_data.append({"teams": group, "played": played_results, "future": future_matches})

assert sum(len(item["played"]) + len(item["future"]) for item in group_data) == 72

In [176]:
def simulate_group(group_info, rng):
    results = list(group_info["played"])
    for home, away, home_rate, away_rate in group_info["future"]:
        home_goals = int(rng.poisson(home_rate))
        away_goals = int(rng.poisson(away_rate))
        results.append((home, away, home_goals, away_goals))
    return rank_group(group_info["teams"], results, rng)


def run_group_stage(group_data, rng):
    standings = [simulate_group(group_info, rng) for group_info in group_data]
    first_place = [table[0]["team"] for table in standings]
    second_place = [table[1]["team"] for table in standings]
    third_place = [table[2] for table in standings]

    third_lottery = {row["team"]: rng.random() for row in third_place}
    third_place.sort(
        key=lambda row: (row["points"], row["gd"], row["gf"], third_lottery[row["team"]]),
        reverse=True,
    )
    qualified = set(first_place + second_place + [row["team"] for row in third_place[:8]])
    assert len(qualified) == 32
    return qualified

In [177]:
N_SIMULATIONS = 10_000
rng = np.random.default_rng(42)
advance_count = {team: 0 for group in groups for team in group}

for _ in range(N_SIMULATIONS):
    for team in run_group_stage(group_data, rng):
        advance_count[team] += 1

advance_probabilities = (
    pd.DataFrame({
        "team": list(advance_count),
        "advance_probability": [count / N_SIMULATIONS for count in advance_count.values()],
    })
    .sort_values("advance_probability", ascending=False)
    .reset_index(drop=True)
)
advance_probabilities

,team,advance_probability
0,United States,1.0000
1,Switzerland,1.0000
2,Canada,1.0000
3,Egypt,1.0000
4,Germany,1.0000
5,Spain,1.0000
6,Morocco,1.0000
7,Brazil,1.0000
8,Japan,1.0000
9,Mexico,1.0000


## Giới hạn còn lại

Đây vẫn là model gọn: số bàn của hai đội được mô phỏng độc lập, chưa có chấn thương/đội hình và dữ liệu không có điểm fair-play. Tuy vậy, kết quả không còn bị quyết định bởi thứ tự của `set`, trận chưa đá không bị coi là hòa, và model triển khai đã được refit bằng toàn bộ dữ liệu đã biết.